# Transformer Implementation

In [ ]:
# ============================================================
# Mini Transformer from scratch (PyTorch, batch_first)
# Covers: Tokenization, Embeddings, Positional Encoding,
# Multi-Head (masked) Self-Attention, Encoder/Decoder blocks,
# Cross-Attention, Stacking, Final Projection, Generation
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# 0) Tokenizer & input preparation
# -------------------------------
# Simple whitespace tokenizer + tiny vocab builder
PAD, SOS, EOS, UNK = "<pad>", "<sos>", "<eos>", "<unk>"

class TinyTokenizer:
    def __init__(self, texts, min_freq=1):
        from collections import Counter
        tokens = []
        for t in texts:
            tokens += t.strip().split()
        freq = Counter(tokens)
        vocab = [PAD, SOS, EOS, UNK] + [w for w,c in freq.items() if c >= min_freq]
        self.stoi = {w:i for i,w in enumerate(vocab)}
        self.itos = {i:w for w,i in self.stoi.items()}
        self.pad_id, self.sos_id, self.eos_id, self.unk_id = \
            self.stoi[PAD], self.stoi[SOS], self.stoi[EOS], self.stoi[UNK]

    def encode(self, text, add_specials=True):
        ids = [self.stoi.get(w, self.unk_id) for w in text.strip().split()]
        if add_specials:
            ids = [self.sos_id] + ids + [self.eos_id]
        return ids

    def decode(self, ids):
        # stop at EOS for display
        toks = []
        for i in ids:
            w = self.itos.get(int(i), UNK)
            if w == EOS: break
            if w not in (SOS, PAD): toks.append(w)
        return " ".join(toks)

# Toy corpus (copy task: target == source)
corpus = [
    "the cat sat",
    "the dog ran",
    "a cat ran",
    "a dog sat",
    "the cat ran fast",
    "the dog sat quietly"
]
tok = TinyTokenizer(corpus)
vocab_size = len(tok.stoi)

def batchify(sentences, tokenizer, max_len=None):
    enc, dec_in, dec_out = [], [], []
    for s in sentences:
        src = tokenizer.encode(s, add_specials=True)            # [SOS ... EOS]
        tgt = tokenizer.encode(s, add_specials=True)            # copy task
        if max_len:
            src = src[:max_len] ; tgt = tgt[:max_len]
        # decoder input is shifted right (starts with SOS)
        dec_in_ids  = tgt[:-1]
        dec_out_ids = tgt[1:]
        enc.append(src)
        dec_in.append(dec_in_ids)
        dec_out.append(dec_out_ids)

    # pad to equal length
    max_src = max(len(x) for x in enc)
    max_tgt = max(len(x) for x in dec_in)
    pad_id = tokenizer.pad_id

    def pad_to(xs, L):
        return [x + [pad_id]*(L-len(x)) for x in xs]

    return (torch.tensor(pad_to(enc, max_src), device=device),
            torch.tensor(pad_to(dec_in, max_tgt), device=device),
            torch.tensor(pad_to(dec_out, max_tgt), device=device))

# -------------------------------
# 1) Positional Encoding (sin/cos)
# -------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        # x: [B, L, D]
        L = x.size(1)
        return x + self.pe[:, :L]

# -------------------------------
# 2) Scaled Dot-Product Attention
# -------------------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q,K,V: [B, heads, Lq/Lk, d_k]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(Q.size(-1))  # [B,h,Lq,Lk]
    if mask is not None:
        # mask: broadcastable to scores; put -inf where masked
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    out = weights @ V  # [B,h,Lq,d_k]
    return out, weights  # return weights for inspection

# -------------------------------
# 3) Multi-Head Attention
# -------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x_q, x_kv, mask=None):
        B, Lq, D = x_q.shape
        Lk = x_kv.shape[1]
        # project
        Q = self.W_q(x_q).view(B, Lq, self.n_heads, self.d_k).transpose(1, 2)  # [B,h,Lq,d_k]
        K = self.W_k(x_kv).view(B, Lk, self.n_heads, self.d_k).transpose(1, 2) # [B,h,Lk,d_k]
        V = self.W_v(x_kv).view(B, Lk, self.n_heads, self.d_k).transpose(1, 2) # [B,h,Lk,d_k]

        # attention (mask broadcast to [B,1,Lq,Lk] if given)
        if mask is not None and mask.dim() == 3:
            mask = mask.unsqueeze(1)
        attn_out, attn_w = scaled_dot_product_attention(Q, K, V, mask)
        # concat heads
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, Lq, self.d_model)  # [B,Lq,D]
        return self.W_o(attn_out), attn_w  # also return weights

# -------------------------------
# 4) Position-wise Feed-Forward
# -------------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=128, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

# -------------------------------
# 5) Encoder Block: (Self-Attn -> Add&Norm) -> (FFN -> Add&Norm)
# -------------------------------
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=128, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Self-attention (encoder attends to itself)
        attn_out, _ = self.mha(x, x, mask=src_mask)
        x = self.norm1(x + self.drop(attn_out))  # Add & Norm
        # Feed-forward
        ff = self.ffn(x)
        x = self.norm2(x + self.drop(ff))        # Add & Norm
        return x

# -------------------------------
# 6) Decoder Block:
#    Masked Self-Attn -> Add&Norm -> Cross-Attn -> Add&Norm -> FFN -> Add&Norm
# -------------------------------
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=128, dropout=0.1):
        super().__init__()
        self.self_mha = MultiHeadAttention(d_model, n_heads)
        self.cross_mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        # 1) Masked self-attention (look-ahead)
        self_attn_out, self_w = self.self_mha(x, x, mask=tgt_mask)
        x = self.norm1(x + self.drop(self_attn_out))
        # 2) Cross-attention (decoder queries encoder outputs)
        cross_attn_out, cross_w = self.cross_mha(x, enc_out, mask=src_mask)
        x = self.norm2(x + self.drop(cross_attn_out))
        # 3) Feed-forward
        ff = self.ffn(x)
        x = self.norm3(x + self.drop(ff))
        return x, self_w, cross_w

# -------------------------------
# 7) Full Encoder / Decoder Stacks
# -------------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len=256, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])

    def forward(self, src, src_mask=None):
        x = self.pos(self.embed(src))
        for layer in self.layers:
            x = layer(x, src_mask)
        return x  # encoder "memory"

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len=256, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, dropout=dropout) for _ in range(n_layers)])

    def forward(self, tgt, enc_out, tgt_mask=None, src_mask=None):
        x = self.pos(self.embed(tgt))
        self_ws, cross_ws = [], []
        for layer in self.layers:
            x, sw, cw = layer(x, enc_out, tgt_mask, src_mask)
            self_ws.append(sw)    # collect self-attn weights
            cross_ws.append(cw)   # collect cross-attn weights
        return x, self_ws, cross_ws

# -------------------------------
# 8) Transformer Seq2Seq
# -------------------------------
class MiniTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=64, n_layers=2, n_heads=4, max_len=256):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, n_layers, n_heads, max_len)
        self.decoder = Decoder(tgt_vocab, d_model, n_layers, n_heads, max_len)
        self.proj = nn.Linear(d_model, tgt_vocab)

    def make_src_mask(self, src, pad_id):
        # [B, 1, 1, L_src] — 1 keeps dims broadcastable for heads
        mask = (src != pad_id).unsqueeze(1).unsqueeze(2)
        return mask

    def make_tgt_mask(self, tgt, pad_id):
        # Padding mask
        pad_mask = (tgt != pad_id).unsqueeze(1).unsqueeze(2)  # [B,1,1,L_tgt]
        # Look-ahead mask
        L = tgt.size(1)
        look_ahead = torch.tril(torch.ones(L, L, device=tgt.device)).bool()  # [L,L]
        look_ahead = look_ahead.unsqueeze(0).unsqueeze(1)  # [1,1,L,L]
        return pad_mask & look_ahead  # both padding and future masking

    def forward(self, src, tgt, pad_id):
        src_mask = self.make_src_mask(src, pad_id)  # encoder padding mask
        tgt_mask = self.make_tgt_mask(tgt, pad_id)  # decoder causal mask
        enc_out = self.encoder(src, src_mask)
        dec_out, self_ws, cross_ws = self.decoder(tgt, enc_out, tgt_mask, src_mask)
        logits = self.proj(dec_out)
        return logits, self_ws, cross_ws

# -------------------------------
# 9) Train briefly on copy task
# -------------------------------
d_model = 64
n_layers = 2
n_heads = 4
model = MiniTransformer(vocab_size, vocab_size, d_model, n_layers, n_heads).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=tok.pad_id)

def train_step(batch_sents):
    model.train()
    src, tgt_in, tgt_out = batchify(batch_sents, tok)
    logits, _, _ = model(src, tgt_in, tok.pad_id)  # [B,L,V]
    loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

for epoch in range(8):
    total = 0.0
    for i in range(0, len(corpus), 2):
        total += train_step(corpus[i:i+2])
    print(f"Epoch {epoch+1}: loss={total:.4f}")

# -------------------------------
# 10) Autoregressive generation
# -------------------------------
@torch.no_grad()
def generate(text, max_new_tokens=12):
    model.eval()
    src, _, _ = batchify([text], tok)
    enc_out = model.encoder(src, model.make_src_mask(src, tok.pad_id))
    # start decoder with SOS
    tgt = torch.tensor([[tok.sos_id]], device=device)
    for _ in range(max_new_tokens):
        tgt_mask = model.make_tgt_mask(tgt, tok.pad_id)
        dec_out, _, _ = model.decoder(tgt, enc_out, tgt_mask, model.make_src_mask(src, tok.pad_id))
        logits = model.proj(dec_out[:, -1:, :])  # last step
        next_id = logits.argmax(dim=-1)          # greedy
        tgt = torch.cat([tgt, next_id], dim=1)
        if int(next_id[0,0]) == tok.eos_id:
            break
    return tok.decode(tgt[0].tolist())

# Try a few generations (should learn to copy)
tests = [
    "the cat sat",
    "a dog ran",
    "the dog sat quietly"
]
for t in tests:
    print(f"\nSRC:  {t}")
    print(f"OUT:  {generate(t)}")


# Transformer Application In Translation

In [ ]:

# from transformers import MarianMTModel, MarianTokenizer

# # English to French model
# model_name = 'Helsinki-NLP/opus-mt-en-fr'
# tokenizer = MarianTokenizer.from_pretrained(model_name)
# model = MarianMTModel.from_pretrained(model_name)

# # Input text
# src_text = input("Enter the text : ")
# tokenized = tokenizer(src_text, return_tensors="pt", padding=True)

# # Generate translation
# translated = model.generate(**tokenized)
# result = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

# print(result)

In [4]:
from transformers import MarianMTModel, MarianTokenizer

model_name_hn = 'Helsinki-NLP/opus-mt-en-hi'
model_name_fr = 'Helsinki-NLP/opus-mt-en-fr'

choice = int(input("Choose the language for translation: 1)French 2)Hindi :"))

if choice == 1:
    tokenizer = MarianTokenizer.from_pretrained(model_name_fr)
    model = MarianMTModel.from_pretrained(model_name_fr)
    src_text = input("Enter English text: ").strip()
    tokenized = tokenizer(src_text, return_tensors="pt", padding=True, truncation=True)

# Generate translation with beam search
    translated = model.generate(**tokenized, max_length=50, num_beams=5, early_stopping=True)
    result = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]

    print("French Translation:", result[0])

elif choice==2:
    tokenizer = MarianTokenizer.from_pretrained(model_name_hn)
    model = MarianMTModel.from_pretrained(model_name_hn)
    src_text = input("Enter English text: ").strip()
    tokenized = tokenizer(src_text, return_tensors="pt", padding=True, truncation=True)

# Generate translation with beam search
    translated = model.generate(**tokenized, max_length=50, num_beams=5, early_stopping=True)
    result = [tokenizer.decode(t, skip_special_tokens=True) for t in translated]
    print("Hindi Translation:", result[0])

else :
    print("Insufficient choice !! choose again")



Hindi Translation: आप कैसे हैं
